# Tech Challenge — Fase 3
## Predição e Inteligência Analítica para Alfabetização no Brasil

### Notebook baseado exclusivamente na camada Gold da Fase 2

O modelo principal procura responder, com os dados disponíveis na Gold:

- quais municípios apresentam maior risco educacional;
- quais variáveis estão mais associadas ao atingimento da meta;
- quais municípios/regiões possuem padrões semelhantes;
- como apoiar a priorização de políticas públicas;
- quando houver histórico temporal suficiente, como usar informações de um ano para prever o atingimento da meta no ano seguinte.

> **Importante:** nenhuma variável derivada diretamente do resultado do mesmo período é utilizada como feature do modelo, evitando data leakage.

## 1. Aderência ao enunciado

O Tech Challenge exige:

1. utilização da camada Gold da Fase 2;
2. análise exploratória;
3. pipeline completa de Machine Learning;
4. imputação e transformação de variáveis;
5. prevenção de data leakage;
6. separação treino/teste e validação;
7. otimização/generalização;
8. Feature Importance e SHAP;
9. aplicação estratégica para municípios e políticas públicas;
10. documentação das limitações e possíveis evoluções.

Este notebook distingue claramente:

- **objetivo conceitual do desafio:** alfabetização;
- **unidade observacional disponível na Gold:** município/ano;
- **target executável com a Gold:** atingimento da meta;
- **não permitido:** inventar registros individuais a partir de agregados municipais.

Essa decisão privilegia rastreabilidade, validade estatística e reprodutibilidade.

In [1]:

# ============================================================
# 2. Configuração e imports
# ============================================================
from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from joblib import dump

warnings.filterwarnings("ignore")

SEED = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
GOLD_DIR = DATA_DIR / "gold"
REPORTS = ROOT / "reports"
FIGURES = ROOT / "images"
MODELS = ROOT / "models"

for p in [REPORTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print("Diretório de execução:", ROOT)
print("Gold:", GOLD_DIR)


Diretório de execução: f:\FIAP\TechChallenge-Fase3
Gold: f:\FIAP\TechChallenge-Fase3\data\gold


## 3. Descoberta e auditoria da Gold

O notebook procura automaticamente os arquivos Parquet/CSV da camada `data/gold`. A escolha é baseada no conteúdo e nos nomes das tabelas, priorizando a Gold analítica municipal.

In [2]:
# ============================================================
# 3. Descoberta e composição da Gold
# ============================================================
def read_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, sep=None, engine="python")

def normalize_col(c):
    c = str(c).strip().lower()
    repl = {
        "ç":"c","ã":"a","á":"a","à":"a","â":"a","ä":"a",
        "é":"e","ê":"e","í":"i","ó":"o","ô":"o","õ":"o","ú":"u"
    }
    for a,b in repl.items():
        c = c.replace(a,b)
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

files = sorted(list(GOLD_DIR.rglob("*.parquet")) + list(GOLD_DIR.rglob("*.csv")))

if not files:
    raise FileNotFoundError("Nenhum arquivo foi encontrado em data/gold.")

inventory = []
tables = {}

for f in files:
    try:
        tmp = read_table(f).copy()
        tmp.columns = [normalize_col(c) for c in tmp.columns]
        tables[str(f)] = tmp

        cols = list(tmp.columns)
        score = 0
        for term, pts in [
            ("resultado_alfabetizacao", 12),
            ("target_atingiu_meta", 12),
            ("meta_alfabetizacao", 8),
            ("gap_meta", 8),
            ("municipio", 6),
            ("uf", 3),
            ("populacao", 3),
            ("socio", 2),
            ("risco", 2),
            ("ano", 2),
        ]:
            if any(term in c for c in cols):
                score += pts

        inventory.append({
            "arquivo": str(f),
            "linhas": len(tmp),
            "colunas": len(tmp.columns),
            "score": score
        })
    except Exception as exc:
        inventory.append({
            "arquivo": str(f),
            "linhas": None,
            "colunas": None,
            "score": -999,
            "erro": str(exc)
        })

inventory_df = pd.DataFrame(inventory).sort_values(
    ["score", "linhas"], ascending=[False, False]
)
display(inventory_df)

# Identificação de chaves comuns.
all_cols = sorted(set().union(*[set(t.columns) for t in tables.values()]))

def first_in(df, patterns):
    for p in patterns:
        hits = [c for c in df.columns if re.search(p, c, re.I)]
        if hits:
            return hits[0]
    return None

def choose_key_pair(left, right):
    candidates = [
        ("id_municipio", "id_municipio"),
        ("codigo_municipio", "codigo_municipio"),
        ("cod_municipio", "cod_municipio"),
        ("municipio_id", "municipio_id"),
        ("uf", "uf"),
    ]
    for a,b in candidates:
        if a in left.columns and b in right.columns:
            return a,b
    return None

# Começa pela tabela Gold com melhor score e tenta enriquecer com as demais
# somente quando existe uma chave de junção segura.
base_path = Path(inventory_df.iloc[0]["arquivo"])
df = tables[str(base_path)].copy()

for other_path, other in tables.items():
    if other_path == str(base_path):
        continue

    # evita joins quando não há chave municipal/ano comum
    common = set(df.columns).intersection(other.columns)
    keys = [k for k in ["id_municipio","codigo_municipio","cod_municipio","municipio_id"] if k in common]

    if not keys:
        continue

    join_keys = keys.copy()
    if "ano" in common:
        join_keys.append("ano")

    # Só agrega tabela externa se a combinação de chaves for suficientemente única.
    right = other.copy()
    if right.duplicated(join_keys).any():
        agg_cols = [c for c in right.columns if c not in join_keys]
        num_cols = [c for c in agg_cols if pd.api.types.is_numeric_dtype(right[c])]
        cat_cols = [c for c in agg_cols if not pd.api.types.is_numeric_dtype(right[c])]

        agg_map = {c:"mean" for c in num_cols}
        agg_map.update({c:"first" for c in cat_cols[:20]})
        if agg_map:
            right = right.groupby(join_keys, as_index=False).agg(agg_map)

    new_cols = [c for c in right.columns if c not in df.columns or c in join_keys]
    right = right[join_keys + [c for c in new_cols if c not in join_keys]]

    if len(right.columns) > len(join_keys):
        df = df.merge(right, on=join_keys, how="left", suffixes=("", "_enriched"))

# Remove colunas duplicadas criadas por joins.
df = df.loc[:, ~df.columns.duplicated()].copy()

print("Tabelas Gold disponíveis:", len(tables))
print("Tabela base:", base_path.name)
print("Dimensões finais após enriquecimento:", df.shape)
print("Colunas:", df.columns.tolist())

,arquivo,linhas,colunas,score
1,f:\FIAP\TechChallenge-Fase3\data\gold\gold_dat...,23995,10,50
0,f:\FIAP\TechChallenge-Fase3\data\gold\gold_com...,23995,10,38
2,f:\FIAP\TechChallenge-Fase3\data\gold\gold_ind...,23995,28,38
3,f:\FIAP\TechChallenge-Fase3\data\gold\gold_ind...,145,41,33


Tabelas Gold disponíveis: 4
Tabela base: gold_dataset_ia_municipio.parquet
Dimensões finais após enriquecimento: (23995, 29)
Colunas: ['ano', 'id_municipio', 'rede', 'serie', 'resultado_alfabetizacao', 'meta_alfabetizacao', 'gap_meta', 'status_meta', 'classificacao_risco', 'target_atingiu_meta', 'ano_meta', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', 'taxa_alfabetizacao_meta_base', 'data_ingestao', 'origem', 'data_processamento', 'camada_origem', 'status_validacao', 'data_processamento_gold']


## 4. Perfil da base e identificação das variáveis

As colunas são normalizadas apenas para facilitar a automação. Os dados originais permanecem como fonte da análise.

In [3]:
# ============================================================
# 4. Perfil da base e identificação das variáveis
# ============================================================
def normalize_col(c):
    c = str(c).strip().lower()
    c = c.replace("ç", "c").replace("ã", "a").replace("á", "a").replace("é", "e")
    c = c.replace("í", "i").replace("ó", "o").replace("ú", "u")
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

df.columns = [normalize_col(c) for c in df.columns]

def first_match(patterns, columns=None):
    columns = list(columns if columns is not None else df.columns)
    for pattern in patterns:
        hits = [c for c in columns if re.search(pattern, c, flags=re.I)]
        if hits:
            return hits[0]
    return None

municipality_col = first_match([
    r"^id_municipio$", r"^codigo_municipio$", r"^cod_municipio$",
    r"municipio", r"nome_municipio"
])
uf_col = first_match([r"^uf$", r"sigla_uf", r"estado"])
year_col = first_match([r"^ano$", r"ano_referencia", r"ano_ref", r"ano_base", r"year"])
result_col = first_match([
    r"^resultado_alfabetizacao$", r"resultado.*alfabet",
    r"taxa.*alfabet", r"percentual.*alfabet"
])
target_col = first_match([
    r"^target_atingiu_meta$", r"atingiu_meta", r"ating.*meta"
])
meta_col = first_match([r"meta_alfabetizacao", r"meta.*alfabet"])
gap_col = first_match([r"^gap_meta$", r"gap.*meta"])
risk_col = first_match([r"classificacao_risco", r"risco"])
population_col = first_match([r"populacao", r"populacao_total", r"pop_"])

identified = {
    "municipio": municipality_col,
    "uf": uf_col,
    "ano": year_col,
    "resultado": result_col,
    "target": target_col,
    "meta": meta_col,
    "gap": gap_col,
    "risco": risk_col,
    "populacao": population_col,
}
print(json.dumps(identified, ensure_ascii=False, indent=2))

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

display(audit)


{
  "municipio": "id_municipio",
  "uf": null,
  "ano": "ano",
  "resultado": "resultado_alfabetizacao",
  "target": "target_atingiu_meta",
  "meta": "meta_alfabetizacao",
  "gap": "gap_meta",
  "risco": "classificacao_risco",
  "populacao": null
}


,dtype,missing,missing_pct,unicos
meta_alfabetizacao,float64,18763,78.20,2745
gap_meta,float64,18763,78.20,4077
target_atingiu_meta,float64,18763,78.20,2
taxa_alfabetizacao_meta_base,float64,12224,50.94,2591
proporcao_aluno_nivel_8,float64,11547,48.12,1794
proporcao_aluno_nivel_3,float64,11547,48.12,2733
proporcao_aluno_nivel_6,float64,11547,48.12,3166
proporcao_aluno_nivel_5,float64,11547,48.12,3259
proporcao_aluno_nivel_4,float64,11547,48.12,2971
proporcao_aluno_nivel_1,float64,11547,48.12,1849


## 5. Definição do target sem vazamento

### Estratégia prioritária: previsão temporal

Se a Gold possuir histórico de pelo menos dois anos por município, o notebook cria:

**`target_futuro = 1`** quando o município atinge a meta no próximo período e **`0`** quando não atinge.

As features são construídas a partir do período atual, permitindo uma interpretação de previsão.

Quando não existe histórico temporal suficiente, o notebook utiliza `target_atingiu_meta` da própria Gold como target municipal, deixando a limitação explícita.

Não são utilizadas como features as variáveis diretamente derivadas do resultado do período previsto, como `resultado_alfabetizacao`, `meta_alfabetizacao`, `gap_meta`, `status_meta`, `classificacao_risco` e o próprio target.

In [4]:
# ============================================================
# 5. Definição do target sem vazamento
# ============================================================

work = df.copy()

if target_col is None:
    raise ValueError(
        "A Gold selecionada não possui target_atingiu_meta/atingiu_meta. "
        "É necessário definir explicitamente um target compatível com a Gold."
    )

work[target_col] = pd.to_numeric(work[target_col], errors="coerce")
work = work.dropna(subset=[target_col]).copy()
work[target_col] = work[target_col].astype(int)

use_future_target = False
future_target_col = "target_futuro"

if municipality_col and year_col:
    work[year_col] = pd.to_numeric(work[year_col], errors="coerce")
    valid_years = work[year_col].dropna().nunique()
    repeated_municipal_years = (
        work.dropna(subset=[municipality_col, year_col])
            .groupby(municipality_col)[year_col]
            .nunique()
    )
    if valid_years >= 2 and (repeated_municipal_years >= 2).mean() >= 0.50:
        ordered = work.sort_values([municipality_col, year_col]).copy()
        ordered[future_target_col] = ordered.groupby(municipality_col)[target_col].shift(-1)
        ordered["ano_alvo"] = ordered.groupby(municipality_col)[year_col].shift(-1)
        ordered = ordered.dropna(subset=[future_target_col]).copy()
        ordered[future_target_col] = ordered[future_target_col].astype(int)
        work = ordered
        TARGET = future_target_col
        use_future_target = True
    else:
        TARGET = target_col
else:
    TARGET = target_col

print("Target utilizado:", TARGET)
print("Estratégia temporal:", use_future_target)
print("Distribuição:")
display(work[TARGET].value_counts(dropna=False).rename("quantidade").to_frame())
display((work[TARGET].value_counts(normalize=True) * 100).round(2).rename("%").to_frame())


Target utilizado: target_atingiu_meta
Estratégia temporal: False
Distribuição:


,quantidade
target_atingiu_meta,
1,2788
0,2444


,%
target_atingiu_meta,
1,53.29
0,46.71
